# 05 · From crawl to knowledge base (RAG)

Crawl the sandbox newsroom, extract articles, drop near-duplicates, chunk, embed and search — then optionally let a language model answer with citations.

*Crawl berita sandbox, ekstrak artikel, buang duplikat, chunk, embed, dan cari — lalu opsional jawab dengan LLM beserta sitasi.*

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.*

In [ ]:
// Packages / Paket
#r "nuget: Gravicode.ScrapyNet, 0.1.0"
#r "nuget: Gravicode.ScrapyNet.Sandbox, 0.1.0"
#r "nuget: Gravicode.ScrapyNet.AI, 0.1.0"

// Before the packages are on nuget.org, point at a local build instead:
//   dotnet pack ScrapyNet.slnx -c Release -o artifacts
// and add this line above the #r lines:  #i "nuget: ../artifacts"

In [ ]:
using ScrapyNet;
using ScrapyNet.AI;
using ScrapyNet.Sandbox;

var site = await SandboxSite.StartAsync();
var kb = new KnowledgeBase(new HashingEmbeddingGenerator(512), chunking: new ChunkingOptions { MaxWords = 120, OverlapWords = 20 });

In [ ]:
public sealed class NewsSpider(string url) : Spider
{
    public override IReadOnlyList<string> StartUrls => [url];

    public override async IAsyncEnumerable<object> Parse(Response response)
    {
        foreach (var r in response.FollowAllCss("article.teaser h2 a", Article)) yield return r;
        await Task.CompletedTask;
    }

    private IEnumerable<object> Article(Response r)
    {
        yield return new Dictionary<string, object?> { ["url"] = r.Url, ["html"] = r.Text };
    }
}

In [ ]:
var settings = new Settings();
settings.Set(SettingKeys.LogLevel, "WARNING");
settings.ItemPipelines.Add(new RagIngestionPipeline(kb) { Summarizer = new ExtractiveSummarizer() }, 800);
await Scrapy.CrawlAsync(new NewsSpider(site.Url("/news/")), settings);
display($"{kb.Stats.Documents} documents, {kb.Stats.Chunks} chunks, {kb.Stats.NearDuplicates} near-duplicate skipped");

In [ ]:
var hits = await kb.SearchAsync("how do crawlers avoid proxy bans?", top: 3);
display(hits.Select(h => $"{h.Score:0.000}  {h.Record.Metadata["title"]}"));

## Optional: answers from a language model

Any OpenAI-compatible endpoint works (OpenAI, Azure OpenAI, DeepSeek, Ollama). Set the key in an environment variable — never in the notebook.

*Endpoint apa pun yang kompatibel OpenAI bisa dipakai. Simpan kunci di environment variable, jangan di notebook.*

In [ ]:
if (Environment.GetEnvironmentVariable("SCRAPYNET_DEEPSEEK_KEY") is { } key)
{
    var withModel = new KnowledgeBase(kb.Embeddings, kb.Store, kb.Chunking)
    {
        LanguageModel = new OpenAiChatModel("deepseek-v4-flash", key, "https://api.deepseek.com/v1/"),
    };
    var answer = await withModel.AskAsync("What does proxy rotation do when a proxy gets banned?");
    display(answer.Answer);
}
await site.DisposeAsync();